# MultiWOZ annotation with Ollama

This notebook runs an Ollama-hosted LLM over MultiWOZ 2.3 dialogs and writes structured JSON annotations.

**Note:** Raw MultiWOZ data should live outside the git repo (e.g., `data/`).

In [ ]:
# Imports
import json
import re
from json import JSONDecodeError
from pathlib import Path
from typing import Any, Dict, List, Optional

from ollama import Client


In [ ]:
# Paths / config (edit these)
INPUT_PATH = Path(r"data/raw/multiwoz23/data.json")   # <-- change to your local path
OUTPUT_PATH = Path(r"data/processed/annotations_ann1.jsonl")

OLLAMA_HOST = "http://localhost:11434"               # e.g. "http://hcai.uni-regensburg.de:11434"
MODEL_NAME = "gemma3:27b"                            # e.g. "llama3:70b"
TIMEOUT_S = 500

# How many dialogs to annotate (None = all)
MAX_DIALOGS: Optional[int] = 100
RANDOM_SEED = 42


In [ ]:
# Load MultiWOZ
with INPUT_PATH.open(encoding="utf-8") as f:
    data = json.load(f)

# MultiWOZ is a dict: dialog_id -> dialog
dialog_items = list(data.items())
print(f"Loaded dialogs: {len(dialog_items):,}")
print("Example keys:", list(data.keys())[:3])


In [ ]:
# Connect to Ollama
client = Client(host=OLLAMA_HOST, timeout=TIMEOUT_S)

# Warm-up call (optional)
_ = client.generate(model=MODEL_NAME, prompt="Say OK.", options={"temperature": 0})
print("Ollama OK:", MODEL_NAME)


## Prompt

The following cell defines the prompt templates and constraints used for annotation.

In [ ]:
from string import Template

ALLOWED_SLOTS = """
Allowed slots by domain (use ONLY these keys; drop unknown keys):
- train: dest, depart, arriveBy, leaveAt, day, people, ref
- hotel: area, stars, parking, wifi, price, nights, people, day, ref
- restaurant: area, food, price, day, time, people, ref
- taxi: depart, dest, leaveAt, arriveBy
- booking: ref
- attraction: area, type, price, name, day, time, ref
"""

CUSTOMER_ALLOWED = """
Allowed domain–topic pairs for CUSTOMER (use ONLY these; never invent new ones):
- Attraction-Inform, Attraction-Request
- Restaurant-Inform, Restaurant-Request
- Hotel-Inform, Hotel-Request
- Train-Inform, Train-Request
- Taxi-Inform, Taxi-Request
- Police-Inform, Police-Request
- Hospital-Inform, Hospital-Request
- general-greet, general-thank, general-bye
"""

EXPERT_ALLOWED = """
Allowed domain–topic pairs for EXPERT (must use one of these):
- Train-OfferBook, Train-OfferBooked, Train-Inform, Train-Request, Train-Select, Train-NoOffer
- Restaurant-Inform, Restaurant-Request, Restaurant-Select, Restaurant-Recommend, Restaurant-NoOffer
- Hotel-Inform, Hotel-Request, Hotel-Select, Hotel-Recommend, Hotel-NoOffer
- Attraction-Inform, Attraction-Request, Attraction-Select, Attraction-Recommend, Attraction-NoOffer
- Taxi-Inform, Taxi-Request
- Hospital-Inform, Hospital-Request
- Police-Inform
- Booking-Book, Booking-Inform, Booking-Request, Booking-NoBook
- general-greet, general-bye, general-welcome, general-reqmore
"""

COMMON_RULES = """
STRICT RULES:
1) Annotate ONLY the LAST_UTTERANCE.
2) Use ONLY allowed slot keys. Do NOT invent keys. Normalize obvious variants.
3) Normalize values: lowercase free text; weekdays → "monday"..."sunday"; numbers → strings.
4) Output exactly ONE JSON list with 1–2 FULL objects for the LAST_UTTERANCE only.
   - Never output a bare slots dict.
   - No prose or markdown.
5) Multi-intent turns (applies to customer + expert):
   - If a turn provides information AND asks a clarifying question → TWO objects.
   - If a CUSTOMER expresses acknowledgement/sentiment (“great”, “excellent”, “okay”, “thanks”)
     AND makes a request → TWO objects:
       • First: general-thank (or general-inform)
       • Second: <Domain>-Request
6) If specific choices/options are presented, topic="select".
7) Respect CUSTOMER_ALLOWED and EXPERT_ALLOWED pairs strictly.
8) Dialog_act format "<Domain>-<ActType>":
   - Domain ∈ {"Train","Hotel","Restaurant","Attraction","Taxi","Hospital","Police","Booking"} or "general".
   - CUSTOMER ActType ∈ {"Inform","Request","greet","thank","bye"}.
   - EXPERT ActType ∈ {"Inform","Request","Select","Recommend","NoOffer","Book","NoBook","greet","bye","welcome","reqmore"}.
   - If domain = "general": act ∈ {"greet","thank","bye","welcome","reqmore"}. eg. "Your Welcome!" → domain="general", topic="thank", dialog_act="general-thank".
8.1) CUSTOMER-specific constraints:
   - Service domains: {"Train","Hotel","Restaurant","Attraction","Taxi","Hospital","Police","Booking"}
     → ActType MUST be one of {"Inform","Request"}.
     → So only these dialog_act values are allowed for CUSTOMER service domains:
       "Train-Inform","Train-Request",
       "Hotel-Inform","Hotel-Request",
       "Restaurant-Inform","Restaurant-Request",
       "Attraction-Inform","Attraction-Request",
       "Taxi-Inform","Taxi-Request",
       "Hospital-Inform","Hospital-Request",
       "Police-Inform","Police-Request",
       "Booking-Inform","Booking-Request".
   - Social “small talk” acts greet/thank/bye MUST ALWAYS use domain="general".
     → Allowed dialog_act values for those are ONLY:
       "general-greet","general-thank","general-bye".
   """

EXPERT_SUPERSEDING_EXCEPTION = """
SUPERSEDING EXCEPTION (TRAIN-ONLY booking phrasing):
If a booking reference/code appears in LAST_UTTERANCE:

  - If it clearly refers to a TRAIN booking (e.g., mentions "train booking",
    "train ticket", platform, departure/arrival time, etc.):
      • domain = "Train"
      • topic  = "OfferBook"      (if offering to book)
      • topic  = "OfferBooked"    (if confirming success)

  - In ANY other case involving a booking reference/code:
      • domain = "Booking"
      • topic  = "Booked"

Notes:
- "OfferBook" and "OfferBooked" MUST ONLY occur with domain="Train".
- Generic booking confirmation is always domain="Booking", topic="Booked".
"""


customer_prompt = Template("""
You annotate MultiWOZ-style dialogues. Output MUST be a JSON list (e.g., [{"example":"value"}]) with 1–2 objects. JSON only.

Schema for EACH object:
{
  "speaker": "customer",
  "domain": "train|hotel|restaurant|attraction|taxi|hospital|police|booking|general",
  "topic": "inform|request|greet|thank|bye|other",
  "slots": {},
  "summary": "one-sentence gist",
  "dialog_act": "<Domain>-<ActType>"
}

Keep strictly to the following rules for CUSTOMER turns.
""" + CUSTOMER_ALLOWED + "\n" + ALLOWED_SLOTS + "\n" + COMMON_RULES + r"""

EXAMPLE OF MULTI-INTENT CUSTOMER TURN:

Utterance:
"Great, thanks! Could you give me the phone number?"

Correct annotation:
[
  {
    "speaker": "customer",
    "domain": "general",
    "topic": "thank",
    "slots": {},
    "summary": "Customer expresses gratitude.",
    "dialog_act": "general-thank"
  },
  {
    "speaker": "customer",
    "domain": "restaurant",
    "topic": "request",
    "slots": {},
    "summary": "Customer asks for the phone number.",
    "dialog_act": "Restaurant-Request"
  }
]

DIALOGUE:
$dialogue

LAST_UTTERANCE:
$last_utterance
""".strip())


expert_prompt = Template("""
You annotate MultiWOZ-style dialogues. Output MUST be a JSON list (e.g., [{"example":"value"}]) with 1–2 objects. JSON only.

Schema for EACH object:
{
  "speaker": "expert",
  "domain": "train|hotel|restaurant|attraction|taxi|hospital|police|booking|general",
  "topic": "inform|request|offer|select|recommend|nooffer|book|nobook|greet|bye|welcome|reqmore|other",
  "slots": {},
  "summary": "one-sentence gist of this system turn",
  "dialog_act": "<Domain>-<ActType>"
}

""" + EXPERT_SUPERSEDING_EXCEPTION + r"""
EXAMPLE OF MULTI-INTENT EXPERT TURN (IMPORTANT):

Utterance:
"Sure, I have several options available. Would you like something in the north or the south?"

Correct annotation:
[
  {
    "speaker": "expert",
    "domain": "hotel",
    "topic": "inform",
    "slots": {},
    "summary": "Provides options.",
    "dialog_act": "Hotel-Inform"
  },
  {
    "speaker": "expert",
    "domain": "hotel",
    "topic": "request",
    "slots": {},
    "summary": "Asks for choice.",
    "dialog_act": "Hotel-Request"
  }
]

""" + EXPERT_ALLOWED + "\n" + ALLOWED_SLOTS + "\n" + COMMON_RULES + r"""

DIALOGUE:
$dialogue

LAST_UTTERANCE:
$last_utterance
""".strip())


## Helpers

Parsing and light validation of model output.

In [ ]:
def conv_llm_output_to_actions(llm_output: str) -> Optional[List[Dict[str, Any]]]:
    """Convert model output to a list of JSON objects.

    The model is instructed to output a JSON list of dicts. In practice, it may
    return malformed JSON. This function tries a strict parse first, then falls
    back to a conservative regex extraction.

    :param llm_output: Raw model output string.
    :return: List of action dicts, or None if parsing fails.
    """
    s = (llm_output or "").strip()

    # Rare stray brace fix observed in earlier runs
    if s.endswith("}]}"):
        s = s[:-1]

    # 1) strict JSON
    try:
        parsed = json.loads(s)
        if isinstance(parsed, list) and all(isinstance(x, dict) for x in parsed):
            return parsed
    except Exception:
        pass

    # 2) regex fallback for list-ish strings
    if s.startswith("[") and s.endswith("]"):
        candidates = re.findall(r"\{[^\{\}]+\}", s[1:-1])
        out: List[Dict[str, Any]] = []
        for c in candidates:
            try:
                out.append(json.loads(c))
            except JSONDecodeError:
                continue
        return out if out else None

    return None


def normalize_action(action: Dict[str, Any]) -> Dict[str, Any]:
    """Normalize fields to reduce downstream errors.

    :param action: Parsed action dict from the LLM.
    :return: Normalized action dict.
    """
    out = dict(action)
    slots = out.get("slots", {})
    out["slots"] = slots if isinstance(slots, dict) else {}
    return out


## Annotation loop

Annotate dialogs and write one JSON object per action to JSONL.

In [ ]:
import random

random.seed(RANDOM_SEED)

# Subsample dialogs if requested
if MAX_DIALOGS is not None:
    dialog_items_run = dialog_items[:MAX_DIALOGS]
else:
    dialog_items_run = dialog_items

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

written = 0

with OUTPUT_PATH.open("w", encoding="utf-8") as out_f:
    for d_index, (dialog_id, dialog) in enumerate(dialog_items_run):
        dialogue_so_far = ""
        customer_speaking = True
        state_hint = "general"

        for turn_index, turn in enumerate(dialog.get("log", [])):
            text = turn.get("text", "")

            if customer_speaking:
                last_utterance = f"customer: {text}\n"
                messages = [
                    {"role": "system", "content": SYS_PRE},
                    {"role": "user", "content": customer_prompt.substitute(
                        dialogue=dialogue_so_far,
                        last_utterance=last_utterance,
                        state_hint=state_hint
                    )}
                ]
            else:
                last_utterance = f"expert: {text}\n"
                messages = [
                    {"role": "system", "content": SYS_PRE},
                    {"role": "user", "content": expert_prompt.substitute(
                        dialogue=dialogue_so_far,
                        last_utterance=last_utterance,
                        state_hint=state_hint
                    )}
                ]

            # Call model
            resp = client.chat(
                model=MODEL_NAME,
                messages=messages,
                options={"temperature": 0}
            )
            llm_output = resp["message"]["content"]

            actions = conv_llm_output_to_actions(llm_output)
            if actions is None:
                # Skip but record a minimal error line if you want
                continue

            for a in actions:
                a = normalize_action(a)
                a["_dialog_id"] = dialog_id
                a["_dialog_index"] = d_index
                a["_turn_index"] = turn_index
                a["_speaker"] = "customer" if customer_speaking else "expert"
                a["_source_text"] = text

                out_f.write(json.dumps(a, ensure_ascii=False) + "\n")
                written += 1

            # Update rolling dialogue text
            dialogue_so_far += ("customer: " if customer_speaking else "expert: ") + text + "\n"
            customer_speaking = not customer_speaking

print(f"Wrote actions: {written:,} -> {OUTPUT_PATH}")


## Quick sanity check

Read a few lines back from the JSONL.

In [ ]:
# Read back a few lines
sample_n = 3
with OUTPUT_PATH.open(encoding="utf-8") as f:
    for i, line in zip(range(sample_n), f):
        obj = json.loads(line)
        print(json.dumps(obj, ensure_ascii=False, indent=2)[:800], "\n")
